In [5]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [50]:
import pandas as pd

df = pd.read_json("../data/comment_generation_results.jsonl", lines=True)
# df = pd.read_json("../data/comment_generation_results_devstral.jsonl", lines=True)

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   patch_id   505 non-null    object
 1   prompt     505 non-null    object
 2   reasoning  505 non-null    object
 3   response   505 non-null    object
dtypes: object(4)
memory usage: 15.9+ KB


In [52]:
import json
import pandas as pd
import re

def parse_response(response):
    if pd.isna(response):
        return []

    response = str(response).strip()

    # Remove markdown code fences
    response = re.sub(r"^```json\s*", "", response, flags=re.IGNORECASE)
    response = re.sub(r"\s*```$", "", response)

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        return []

    return data.get("comments", [])


rows = []

for _, row in df.iterrows():
    comments = parse_response(row["response"])

    for comment in comments:
        rows.append({
            "patch_id": row["patch_id"],
            "comment": comment.get("comment"),
            "category": comment.get("category"),
            "severity": comment.get("severity")
        })

result_df = pd.DataFrame(rows)

In [53]:
import uuid

result_df.insert(
    0,
    "comment_id",
    [str(uuid.uuid4()) for _ in range(len(result_df))]
)

In [56]:
result_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   comment_id         448 non-null    object
 1   patch_id           448 non-null    object
 2   comment            448 non-null    object
 3   category           447 non-null    object
 4   severity           448 non-null    object
 5   generation_system  448 non-null    object
dtypes: object(6)
memory usage: 21.1+ KB


In [55]:
result_df["generation_system"] ="qwen3.6-35b-a3b"

In [57]:
df_comments = pd.concat(
    [result_df, df_comments],
    axis=0,
    ignore_index=True
)

In [59]:
df_comments.to_csv("../data/df_n3_all_models.csv", index=False)